# Menu Targeting — User Segments Export
將 8 個 targeting group 嘅 UserID 寫出 Excel，每個 Group 一個 Sheet。

| Sheet Name | Group | Promo |
|------------|-------|-------|
| Premium User | Gp14 | $1600-120 |
| Booking Standing 5 Star | Gp11 | $800-60 |
| Sleeping User (A) | Gp1 — 有買 menu May-Oct 2025，但 Nov 2025-Apr 2026 冇買 | $400-30 |
| Warm User | Gp5 — Apr 2026 有買 menu | $400-30 |
| New User(MAR) | Gp6 — 第一次買 menu 係 Mar 2026，Apr 2026 冇買 | $400-30 |
| Sleeping User (B) | Gp12 — 有買 menu May 2024-Apr 2025，但過去 12 個月冇買 | $400-30 |
| NEW NEW User (A) | Gp13A — 有 VOU txn 單筆 $200+，但冇買過 menu | $400-30 |
| NEW NEW User (B) | Gp13B — 有 VOU txn 喺 bookable POI，但冇買過 menu | $400-30 |

# Menu Targeting — User Segments Export 14_09_2026
將 8 個 targeting group 嘅 UserID 寫出 Excel，每個 Group 一個 Sheet。

| Sheet Name | Group | Promo |
|------------|-------|-------|
| Sleeping User (A) | Gp1 — 有買 menu Sep 2025-Feb 2026，但 Mar-Aug 2026 冇買 | $400-30 |
| Warm User | Gp2 — Aug 2026 有買 menu | $400-30 |
| NEW User (of previous mos) | Gp3 — 第一次買 menu 係 Jul 2026，Aug 2026 冇買 | $400-30 |
| Booking Standing 5 Star | Gp4 | $800-60 |
| Sleeping User (B) | Gp5 — 有買 menu Sep 2024-Aug 2025，但過去 12 個月 (Sep 2025-Aug 2026) 冇買 | $400-30 |
| NEW NEW User (A) | Gp6 — 有 VOU txn 單筆 $200+，但冇買過 menu (Sep 2025-Aug 2026) | $400-30 |
| NEW NEW User (B) | Gp7 — 有 VOU txn 喺 bookable POI，但冇買過 menu (Sep 2025-Aug 2026) | $400-30 |
| Premium User | Gp8 — Sep 2025-Aug 2026 | $1600-120 |

In [3]:
# Install dependencies if needed
# !pip install pymssql pandas openpyxl
import pymssql
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import datetime

print('Libraries loaded OK')

Libraries loaded OK


In [4]:
# ─── DB Connection ────────────────────────────────────────────
DB_SERVER   = '192.168.61.119'
DB_PORT     = 7622
DB_USER     = 'BAReporting'
DB_PASSWORD = 'KeHeCReme8he'

OUTPUT_PATH = r'Menu_Targeting_UserSegments.xlsx'

conn = pymssql.connect(
    server=DB_SERVER, port=DB_PORT,
    user=DB_USER, password=DB_PASSWORD,
    database='Mars', charset='UTF-8'
)
print('DB connected')

DB connected


## Step 1 — 建立共用 Temp Tables 及查詢各 Group

In [ ]:
# ─── 定義每個 Group 嘅 SQL ────────────────────────────────────
# 每條 SQL 係獨立嘅 (使用 CTE/subquery)，唔依賴 temp table
# 返回欄位：userid
# 所有 query 都加咗 User.status = 10 (active user)

# Sheet 出場順序 (跟參考 Excel)
SHEET_ORDER = ['Gp1', 'Gp2', 'Gp3', 'Gp4', 'Gp5', 'Gp6', 'Gp7', 'Gp8']

groups = {}

# ── Gp1: Sleeping User (A) ──────────────────────────────────
groups['Gp1'] = {
    'sheet_name': 'Sleeping User (A)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        may_oct AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2025-09-01' AND PaymentTime < '2026-03-01'
        ),
        nov_apr AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2026-03-01' AND PaymentTime < '2026-09-01'
        )
        SELECT DISTINCT g.userid
        FROM may_oct g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.userid NOT IN (SELECT userid FROM nov_apr)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp2: Warm User ───────────────────────────────────────────
groups['Gp2'] = {
    'sheet_name': 'Warm User',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT g.userid
        FROM menu_buyers g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.PaymentTime >= '2026-08-01' AND g.PaymentTime < '2026-09-01'
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp3: New User (of previous mos) ──────────────────────────────────────
groups['Gp3'] = {
    'sheet_name': 'New User(of previous mos)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        first_buy AS (
            SELECT userid, MIN(PaymentTime) AS first_buy_time FROM menu_buyers GROUP BY userid
        ),
        apr_buyers AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2026-07-01' AND PaymentTime < '2026-08-01'
        )
        SELECT DISTINCT g.userid
        FROM first_buy g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.first_buy_time >= '2026-06-01' AND g.first_buy_time < '2026-07-01'
          AND g.userid NOT IN (SELECT userid FROM apr_buyers)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp11: Booking Standing 5 Star ────────────────────────────
groups['Gp11'] = {
    'sheet_name': 'Booking Standing 5 Star',
    'promo':   '$800-60',
    'sql': """
        USE Mars;
        SELECT DISTINCT u.userid
        FROM [openrice3].[dbo].[User] ou (NOLOCK)
        INNER JOIN [Mars].[dbo].[user] u (NOLOCK) ON u.SSOUserId = ou.SSOUserId
        WHERE ou.UserStar = 5
          AND u.status = 10
        ORDER BY u.userid
    """
}

# ── Gp12: Sleeping User (B) ──────────────────────────────────
groups['Gp12'] = {
    'sheet_name': 'Sleeping User (B)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        prev_year AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2024-08-01' AND PaymentTime < '2025-08-01'
        ),
        last_12m AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2025-08-01' AND PaymentTime < '2026-08-01'
        )
        SELECT DISTINCT g.userid
        FROM prev_year g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.userid NOT IN (SELECT userid FROM last_12m)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp13A: NEW NEW User (A) ──────────────────────────────────
groups['Gp13A'] = {
    'sheet_name': 'NEW NEW User (A)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH vou_txn AS (
            SELECT u.userid, vo.FinalPrice
            FROM VoucherOrder vo (NOLOCK)
            INNER JOIN [mars].[dbo].[User] u (NOLOCK) ON u.SSOUserId = vo.SSOUserId
            WHERE vo.status = 10
              AND vo.PaymentTime >= '2025-08-01' AND vo.PaymentTime < '2026-08-01'
              AND u.status = 10
        ),
        menu_buyers_12m AS (
            SELECT DISTINCT b.userid
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND bp.PaymentTime >= '2025-08-01' AND bp.PaymentTime < '2026-08-01'
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT g.userid
        FROM vou_txn g
        WHERE g.FinalPrice >= 200
          AND g.userid NOT IN (SELECT userid FROM menu_buyers_12m)
        ORDER BY g.userid
    """
}

# ── Gp13B: NEW NEW User (B) ──────────────────────────────────
groups['Gp13B'] = {
    'sheet_name': 'NEW NEW User (B)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH bookable_poi AS (
            SELECT DISTINCT poiid
            FROM BizService (NOLOCK)
            WHERE ServiceTypeId IN (1, 101)
              AND status = 10
              AND ServiceStartTime <= '2025-08-01'
              AND ServiceEndTime   >= '2026-08-01'
        ),
        redeem_poi AS (
            SELECT DISTINCT ow.OfferId
            FROM [Mars].[dbo].[OfferWallet] ow (NOLOCK)
            INNER JOIN [mars].[dbo].[VoucherOrder] vo (NOLOCK) ON vo.OfferId = ow.OfferId
            WHERE ow.RedeemPoiId IS NOT NULL
              AND vo.PaymentTime >= '2025-08-01' AND vo.PaymentTime < '2026-08-01'
              AND EXISTS (SELECT 1 FROM bookable_poi bp WHERE bp.poiid = ow.RedeemPoiId)
        ),
        menu_buyers_12m AS (
            SELECT DISTINCT b.userid
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND bp.PaymentTime >= '2025-08-01' AND bp.PaymentTime < '2026-08-01'
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT u.userid
        FROM VoucherOrder vo (NOLOCK)
        INNER JOIN redeem_poi rpl ON vo.OfferId = rpl.OfferId
        INNER JOIN [mars].[dbo].[User] u (NOLOCK) ON u.SSOUserId = vo.SSOUserId
        WHERE vo.PaymentTime >= '2025-08-01' AND vo.PaymentTime < '2026-08-01'
          AND vo.status = 10
          AND u.status = 10
          AND NOT EXISTS (SELECT 1 FROM menu_buyers_12m m WHERE m.userid = u.userid)
        ORDER BY u.userid
    """
}

# ── Gp14: Premium User ───────────────────────────────────────
groups['Gp14'] = {
    'sheet_name': 'Premium User',
    'promo':   '$1600-120',
    'sql': """
        USE Mars;
        SELECT DISTINCT b.userid
        FROM BookingMenuOrder bmo (NOLOCK)
        INNER JOIN BookingMenu bm (NOLOCK) ON bm.BookingMenuId = bmo.BookingMenuId
        INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
        INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = b.userid
        WHERE b.status = 10 AND bmo.status = 15
          AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
          AND bm.BookingMenuType = 2
          AND bp.PaymentTime >= '2025-08-01' AND bp.PaymentTime < '2026-08-01'
          AND b.userid IS NOT NULL AND b.userid != 0
          AND u.status = 10
        ORDER BY b.userid
    """
}

print(f'{len(groups)} groups defined')
print(f'Sheet order: {[groups[g]["sheet_name"] for g in SHEET_ORDER]}')

8 groups defined
Sheet order: ['Premium User', 'Booking Standing 5 Star', 'Sleeping User (A)', 'Warm User', 'New User(MAR)', 'Sleeping User (B)', 'NEW NEW User (A)', 'NEW NEW User (B)']
